# Build a ReAct agent

You will build a **finance research assistant**. You ask it something like *"Is there any recent
news on AAPL, and is today a weekday?"* and it works out, on its own, that it needs two different
lookups: the news for a ticker, and today's date. It calls both, then answers in plain language.

The interesting part is where the model call goes. **Straight to the model provider, with your
own key.** The gateway is not involved at all. AcruxCore still stores the prompt, the tool catalog and the
traces — but it never sees the completion.

That path has a name: **BYO**, bring your own key. It changes exactly one thing, and that one
thing is the whole lesson of this notebook. Step 1 explains it before any code.

| Piece | What it does | Who runs it |
|---|---|---|
| `finance_research` | the tool in the catalog: a name, a description, one `ticker_symbol` argument | the platform stores it |
| `get_todays_date` | a second tool, with **no arguments at all** | the platform stores it |
| `finance_research_impl` and `todays_date_impl` | the Python that really does the two lookups | **your code** |
| `react-agent-finance` | the prompt: the reasoning instructions, a `question` variable, both tools bound | the platform stores it |
| the loop in Step 9 | render, call OpenAI, report the span, run the tool, report that span, repeat | **your code** |

Every cell runs against a real account, a real provider key, and the real Yahoo Finance news
endpoint. The run saved in this file used OpenAI; the same cells run against any
OpenAI-compatible provider, and Step 0 shows what to change.

**No gateway, and therefore no registered model.** Every other tutorial binds a `model` to the
prompt version, and that name comes from the gateway's model registry. This page never touches
the gateway, so the prompt's `model` field stays `null` and the model id lives in your code. That
is not an oversight; Step 5 says why.

**Two ways to do every step.** Each step that creates something has two headings:
**In the dashboard**, with the values to type, and **The same thing in code**, with a cell to
run. They are not two different features — the dashboard and these calls hit the same API, so the
result is identical. Pick either. Doing both is harmless, because every code cell looks for what
already exists before it creates anything.

**Three kinds of code cell.** Most of this notebook is not the thing you would ship. Every cell's
lead-in says which kind it is:

| Label | What it is | Goes in your app? |
|---|---|---|
| **Setup** | creates something on the platform, once. The dashboard does the same job. | no |
| **Your app** | the code that would really ship | **yes** |
| **Check** | proves the step worked, or shows what just happened | no |
| **Broken on purpose** | a failure being demonstrated | no |

**Companion page:** [Build a ReAct agent](https://docs.acruxcore.com/docs/tutorials/build-a-react-agent)

---

## Step 0 — What you need before you start

**1. An AcruxCore account and a personal API key.** **Account & keys → New key**, named
`react-agent`. Copy it the moment it appears — that is the only time the full value is shown.

**2. A provider key, from any OpenAI-compatible API.** Your code makes the completion call
here, so the only thing that decides who answers is the base URL you point it at. Set the URL
and the model together and nothing else changes:

| Provider | `PROVIDER_BASE_URL` | A `MODEL` it serves |
|---|---|---|
| OpenAI | `https://api.openai.com/v1` | `gpt-4o-mini` |
| OpenRouter | `https://openrouter.ai/api/v1` | `meta-llama/llama-3.3-70b-instruct` |
| Together, Groq, Fireworks | the provider's own `/v1` URL | one of its ids |
| a local server (vLLM, Ollama, LM Studio) | `http://localhost:8000/v1` | whatever it loaded |

The run saved in this file used OpenAI, so that is the default. Two things vary by provider and
are worth knowing before you swap: the model ids are not shared, and each provider words its
error bodies differently, which Step 11 relies on.

**3. No credential and no model.** Skip them. Those exist to let the gateway call a provider on
your behalf, and nothing here goes through the gateway.

**4. `requests`.** That is the only install.

The companion page's Python script reaches Yahoo Finance through
`langchain_community`'s `YahooFinanceNewsTool`. This notebook calls the same endpoint directly
with `requests` instead, for two reasons: it is one dependency instead of three, and you get to
see the two HTTP calls that wrapper is making for you.

In [ ]:
%pip install -q --upgrade requests

**Setup.** Set both keys and name the things this notebook will create.

Two keys, going to two different places. Keeping them in separate variables is not
tidiness — Step 11 shows what happens when one is sent to the wrong host.

`PROVIDER_URL` is the whole of the provider choice. `PROVIDER_HOST` is derived from it rather
than typed, because that string goes on the trace span as the host you really called — typing it
a second time is how a span ends up naming a provider you stopped using.

A key typed into a notebook is saved *inside the notebook file*. Prefer setting these in your
shell before you start Jupyter, and treat this cell as a fallback.

In [1]:
import json
import os
from urllib.parse import urlsplit

# Better: export these in your shell before starting Jupyter.
os.environ.setdefault("ACRUXCORE_API_KEY", "acx_sk_...")
os.environ.setdefault("ACRUXCORE_BASE_URL", "https://api.acruxcore.com/api/v1")
os.environ.setdefault("PROVIDER_API_KEY", "sk-...")
os.environ.setdefault("PROVIDER_BASE_URL", "https://api.openai.com/v1")

ACRUX_KEY = os.environ["ACRUXCORE_API_KEY"]
ACRUX_URL = os.environ["ACRUXCORE_BASE_URL"].rstrip("/")

# BYO: this URL is called directly, never through AcruxCore. Any OpenAI-compatible
# provider works - change the URL and the model together, and nothing else here changes.
PROVIDER_KEY = os.environ["PROVIDER_API_KEY"]
PROVIDER_URL = os.environ["PROVIDER_BASE_URL"].rstrip("/")
PROVIDER_HOST = urlsplit(PROVIDER_URL).netloc   # derived, because it goes on the span

MODEL = os.environ.get("PROVIDER_MODEL", "gpt-4o-mini")   # the provider's own id
PROMPT = "react-agent-finance"     # the prompt this notebook creates
NEWS_TOOL = "finance_research"     # tool 1: takes a ticker symbol
DATE_TOOL = "get_todays_date"      # tool 2: takes nothing

# Do NOT print ACRUX_URL: the saved output would publish whatever host you ran against.

### Preflight

**Check.** Three things can be wrong before anything interesting happens, and they fail in this
order: your AcruxCore key, your OpenAI key, and Yahoo's news endpoint. Checking them separately
means one clear line instead of a stack trace out of the middle of the loop.

The OpenAI check lists models rather than completing anything, so it costs nothing.

In [2]:
import requests

acrux = requests.Session()
acrux.headers.update({
    "Authorization": f"Bearer {ACRUX_KEY}",
    "Content-Type": "application/json",
})


def api(method: str, path: str, body: dict | None = None) -> dict:
    """One JSON call against the AcruxCore API, raising on any non-2xx.

    A notebook helper, NOT an SDK. It only saves repeating the base URL and the
    raise_for_status() line.
    """
    res = acrux.request(method, f"{ACRUX_URL}{path}", json=body, timeout=60)
    res.raise_for_status()
    return res.json() if res.content else {}


# 1. Does the AcruxCore key work?
api("GET", "/prompts?limit=1")
print("acruxcore key: ok")

# 2. Does the provider key work? Listing models costs nothing, and every
# OpenAI-compatible API serves this route.
res = requests.get(f"{PROVIDER_URL}/models",
                   headers={"Authorization": f"Bearer {PROVIDER_KEY}"},
                   timeout=30)
print(f"provider {PROVIDER_HOST}:", "ok" if res.ok else f"FAILED {res.status_code}")
if res.ok:
    print(f"MODEL {MODEL!r} available:", any(m["id"] == MODEL for m in res.json()["data"]))

# 3. Is Yahoo's news endpoint reachable from here? The tool in Step 7 needs it.
yahoo = requests.Session()
yahoo.headers.update({"User-Agent": "Mozilla/5.0"})
yahoo.get("https://fc.yahoo.com", timeout=15)      # sets the session cookie
probe = yahoo.post(
    "https://finance.yahoo.com/xhr/ncp?queryRef=latestNews&serviceKey=ncp_fin",
    json={"serviceConfig": {"snippetCount": 1, "s": ["AAPL"]}}, timeout=20,
)
print("yahoo finance news:", "ok" if probe.ok else f"FAILED {probe.status_code}")

acruxcore key: ok
provider api.openai.com: ok
MODEL 'gpt-4o-mini' available: True
yahoo finance news: ok


---

## Step 1 — Who records the model call

### The general problem

An observability platform can only see what passes through it. That is not a limitation of one
product; it is what "passes through" means. So any platform offering both a proxy and a
bring-your-own-key mode has to answer one question: when the call does *not* pass through us, who
writes it down?

### Where our case sits

AcruxCore has two paths, and they differ in exactly one place.

| | Gateway path | BYO path, this notebook |
|---|---|---|
| who calls the provider | AcruxCore | **your code** |
| who writes the `llm` span | AcruxCore, automatically | **your code** |
| who writes the `tool` span | your code | your code |
| `costUsd` on the span | filled in, from the model's prices | empty — nobody saw the pricing |
| what the prompt's `model` field means | a name in the gateway's registry | unused; the id is in your code |

The tool span was always yours. A client tool runs in your process, so nobody else can report it.
What changes on the BYO path is that the **model** call is now yours to report too.

### The direct answer

You report it with `POST /traces`, one span per model turn, with `kind: "llm"`. Five fields carry
the weight:

| Field | Why it matters |
|---|---|
| `traceId` | the same id on every span, or the run is not one run |
| `model` | which model actually answered — take it from the provider's response, not your constant |
| `provider` | the host you really called, taken from your base URL. This is how a reader knows it bypassed the gateway |
| `usage` | prompt, completion and total tokens, straight from the provider's `usage` object |
| `promptVersionId` | links the run back to the exact prompt version that produced it |

### The trap

Skip the `llm` span and nothing fails. Your agent still answers. The trace still exists, because
the tool spans created it. It just shows tools running with no model turns anywhere — a run that
looks like it happened by itself. Step 11 does this on purpose.

### The recommendation

Report the span. If you use the Python SDK, you do not write any of this by hand: pass
`provider={"apiKey": ..., "baseUrl": ...}` and `chat()` calls the provider directly, mints a trace
id, and reports the span for you. Write it out once, here, so you know what that option is doing.

---

## Step 2 — Create the `finance_research` shell

A tool in the catalog is **two objects**, not one. The **shell** owns the name and the
model-facing description. A **version** owns the argument schema and the executor. This step
creates the shell only.

A shell on its own is not callable. Step 3 makes it callable.

### In the dashboard

**Gateway → Tools → New tool.**

| Field | What to enter |
|---|---|
| **Name** | `finance_research` |
| **Description** | `Search Yahoo Finance news for a ticker symbol.` |

Those are the only two fields. Click **Create tool**.

The description is the sentence the **model** reads when it decides whether to call this tool.
Write it for the model, not for your teammates.

### The same thing in code

**Setup.** `POST /tools`. Find-or-create: it looks the name up first, so a second run of this
notebook creates nothing.

In [3]:
NEWS_DESCRIPTION = "Search Yahoo Finance news for a ticker symbol."


def find_by_name(collection: str, name: str) -> dict | None:
    """The row in /tools or /prompts with exactly this name, or None.

    A notebook helper, NOT an SDK function. `?search=` matches substrings, so the
    exact-name filter has to happen here.
    """
    found = api("GET", f"/{collection}?search={name}&limit=100")
    return next((row for row in found["data"] if row["name"] == name), None)


news_tool = find_by_name("tools", NEWS_TOOL)
if news_tool is None:
    news_tool = api("POST", "/tools", {"name": NEWS_TOOL, "description": NEWS_DESCRIPTION})
    print(f"created tool shell {NEWS_TOOL}")
else:
    print(f"tool {NEWS_TOOL} already in the catalog")

print("tool id:", news_tool["id"])

created tool shell finance_research
tool id: 48e33c73-c789-408e-bcc3-281bbbcd4d54


---

## Step 3 — Commit its version 1, with a client executor

Now the shell gets its argument schema and its executor. `{"type": "client"}` is the sentence
"my own code runs this."

That is the only honest choice here. The other option, HTTP, has the gateway make a request you
describe — but this tool needs two calls in sequence, a cookie carried between them, and a
browser-shaped `User-Agent`. That is a function, not a request.

### In the dashboard

**Gateway → Tools → `finance_research` → New version.**

| Field | What to enter |
|---|---|
| **Parameters** | one row: name `ticker_symbol`, type `string`, **required** |
| **`ticker_symbol` description** | `Stock ticker symbol, e.g. AAPL.` |
| **Executor** | **Client — the caller's app runs it** |

Click **Commit version**. The first version automatically gets the `production` and `staging`
aliases. Later versions move no alias for you — you promote them yourself.

### The same thing in code

**Setup.** `POST /tools/<id>/versions`. A version is immutable, which is why changing a schema
means committing a new version rather than editing this one.

In [4]:
TICKER_SCHEMA = {
    "type": "object",
    "properties": {
        "ticker_symbol": {
            "type": "string",
            "description": 'Stock ticker symbol, e.g. "AAPL".',
        }
    },
    "required": ["ticker_symbol"],
}

if api("GET", f"/tools/{news_tool['id']}/versions?limit=1")["total"] == 0:
    version = api("POST", f"/tools/{news_tool['id']}/versions", {
        "description": NEWS_DESCRIPTION,
        "parametersSchema": TICKER_SCHEMA,
        "executor": {"type": "client"},
    })
    aliases = [a["alias"] for a in version["aliases"]]
    print(f"committed v{version['versionNumber']}, aliases now pointing here: {aliases}")
else:
    print("tool already has a version - nothing committed")

committed v1, aliases now pointing here: ['production', 'staging']


---

## Step 4 — Create the second tool, which takes nothing

`get_todays_date` needs no arguments. A model cannot work out today's date from its own weights —
it was trained in the past — so this is a real tool, and it is the smallest possible one.

An argument-free tool still needs a schema. It is `{"type": "object", "properties": {}}`: an
object with no fields. Leaving the schema out entirely is not the same thing, and some providers
reject it.

You did both calls by hand in Steps 2 and 3, so from here on one helper does the pair.

### In the dashboard

**Gateway → Tools → New tool**, then **New version** on it.

| Field | What to enter |
|---|---|
| **Name** | `get_todays_date` |
| **Description** | `Get today's date.` |
| **Parameters** | none — add no rows at all |
| **Executor** | **Client — the caller's app runs it** |

### The same thing in code

**Setup.** The helper below is find-or-create for both halves, so it is safe to re-run and it
prints which half it skipped.

In [5]:
def create_tool_if_missing(name: str, *, description: str, schema: dict) -> dict:
    """Create the shell, then commit version 1 - but only where they are missing.

    A notebook helper, NOT an SDK function. It wraps the same two real calls made by hand
    in Steps 2 and 3, POST /tools and POST /tools/<id>/versions, and skips whichever
    already exists so this notebook can be re-run. Every tool it makes is a client
    executor, because that is the only kind this notebook uses.
    """
    tool = find_by_name("tools", name)
    if tool is None:
        tool = api("POST", "/tools", {"name": name, "description": description})
        print(f"  + created shell {name}")
    else:
        print(f"  = shell {name} already in the catalog")

    if api("GET", f"/tools/{tool['id']}/versions?limit=1")["total"] == 0:
        version = api("POST", f"/tools/{tool['id']}/versions", {
            "description": description,
            "parametersSchema": schema,
            "executor": {"type": "client"},
        })
        print(f"  + committed {name} v{version['versionNumber']}")
    else:
        print(f"  = {name} already has a version")
    return tool


date_tool = create_tool_if_missing(
    DATE_TOOL,
    description="Get today's date.",
    schema={"type": "object", "properties": {}},     # an object with no fields
)
print("\ntool id:", date_tool["id"])

  + created shell get_todays_date
  + committed get_todays_date v1

tool id: f4d722af-ca1a-4b2d-a404-5ce043297c2e


**Check.** What the model will actually read, for both tools at once. `POST /tools/resolve` takes
a list, and answers in the order you sent it.

Look at the second `parameters` block. An empty `properties` is what "this tool takes nothing"
looks like on the wire.

In [6]:
resolved = api("POST", "/tools/resolve", {"refs": [
    {"name": NEWS_TOOL, "alias": "production"},
    {"name": DATE_TOOL, "alias": "production"},
]})

for entry in resolved["data"]:
    print(f"{entry['function']['name']}  executor={entry['executorType']}  "
          f"v{entry['versionNumber']}")
    print("  parameters:", json.dumps(entry["function"]["parameters"]))

finance_research  executor=client  v1
  parameters: {"type": "object", "required": ["ticker_symbol"], "properties": {"ticker_symbol": {"type": "string", "description": "Stock ticker symbol, e.g. \"AAPL\"."}}}
get_todays_date  executor=client  v1
  parameters: {"type": "object", "properties": {}}


---

## Step 5 — Create the prompt

The system message is what turns two tools into a ReAct-style agent. ReAct means *reason, then
act*: the instructions tell the model to think about what the question actually needs before
calling anything, name both tools, and say when each one is the right choice.

That last part matters more than it looks. The model chooses tools from their names, their
descriptions and this message. Nothing else.

**And no model is bound.** Every other tutorial sets a default model on the prompt version. That
field holds a public name from the gateway's model registry, and this page never touches the
gateway, so it stays `null` here and `MODEL` in your code is the model id instead. If you later
move this agent onto the gateway, that is the field to fill in and the constant to delete.

### In the dashboard

**Prompts → New prompt**, then the **Editor** tab.

| Field | What to enter |
|---|---|
| **Name** | `react-agent-finance` |
| **Description** | `ReAct-style finance research agent, called directly against OpenAI.` |
| **Default model** | leave it **empty** — see above |
| **System message** | the `SYSTEM` string in the next code cell; it is too long to repeat here, and a second copy would drift |
| **User message** | `{{ question }}` |

Click **Commit version**. Because it is the first commit, `production` points at `v1`
automatically.

### The same thing in code

**Setup.** Two separate checks on purpose. A prompt shell with zero versions is a real state — an
earlier run that died between the two calls leaves one — and "the name exists" is not "it has
content".

In [7]:
SYSTEM = (
    "You are a financial research assistant. Reason step by step about what the question "
    "actually needs before answering. Use the finance_research tool to look up recent Yahoo "
    "Finance news for a stock ticker, and the get_todays_date tool whenever the question "
    "depends on today's date (relative dates, whether markets are open, and similar). Only "
    "call a tool when its result is genuinely needed, then give a clear final answer grounded "
    "in what the tools returned."
)

prompt = find_by_name("prompts", PROMPT)
if prompt is None:
    prompt = api("POST", "/prompts", {
        "name": PROMPT,
        "description": "ReAct-style finance research agent, called directly against OpenAI.",
    })
    print(f"created prompt shell {PROMPT}")
else:
    print(f"prompt {PROMPT} already exists")

if api("GET", f"/prompts/{prompt['id']}/versions?limit=1")["total"] == 0:
    pv = api("POST", f"/prompts/{prompt['id']}/versions", {
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": "{{ question }}"},
        ],
        # No "model" key at all: this prompt is never rendered for the gateway.
    })
    print(f"committed prompt v{pv['versionNumber']}, variables={pv['variables']}, "
          f"model={pv['model']}")
else:
    print("prompt already has a version - nothing committed")

created prompt shell react-agent-finance
committed prompt v1, variables=['question'], model=None


---

## Step 6 — Connect both tools to the prompt

A binding joins a tool to a prompt, so one render call returns the messages *and* both tool
schemas together. It is a live setting rather than part of a version, so there is nothing to
commit afterwards.

The binding stores an **alias**, not a version number. Point it at `production` and the prompt
follows whatever you promote to `production` later, with no code change.

### In the dashboard

**Prompts → `react-agent-finance` → Tools tab → + Connect a tool from the catalog**, twice.

| Field | What to enter |
|---|---|
| **First tool** | `finance_research`, alias `production` |
| **Second tool** | `get_todays_date`, alias `production` |
| **Column** | **default** — every alias of the prompt inherits both |

### The same thing in code

**Setup.** `PUT`, not `POST`, and that matters: a `PUT` replaces the binding for that tool rather
than adding a second one, so running this twice leaves two rows and not four.

In [8]:
for tool in (news_tool, date_tool):
    binding = api("PUT", f"/prompts/{prompt['id']}/tools/{tool['id']}",
                  {"tool_alias": "production"})
    print(f"bound {binding['toolName']} @ {binding['toolAlias']} -> "
          f"v{binding['resolvedVersionNumber']}")

bound finance_research @ production -> v1
bound get_todays_date @ production -> v1


**Check.** One render call, and everything the loop needs comes back together. Three things to
notice in the output.

`tools` is already in OpenAI's exact `tools[].function` shape, so there is nothing to reformat
before sending it on. `model` is `None`, as designed. And `versionId` is the value you will stamp
onto every `llm` span, which is what links the trace back to this exact version.

In [9]:
QUESTION = "Is there any recent news on AAPL, and is today a weekday?"

rendered = api("POST", f"/prompts/{PROMPT}/production/render", {"variables": {"question": QUESTION}})

print("model:     ", rendered["model"], "(null on purpose - no gateway here)")
print("versionId: ", rendered["versionId"])
print("tools:     ", [t["function"]["name"] for t in rendered["tools"]])
for message in rendered["messages"]:
    print(f"  {message['role']}: {message['content'][:90]}")

model:      None (null on purpose - no gateway here)
versionId:  2807da05-b3bb-4032-96ad-6ebdc2c7be77
tools:      ['finance_research', 'get_todays_date']
  system: You are a financial research assistant. Reason step by step about what the question actual
  user: Is there any recent news on AAPL, and is today a weekday?


---

## Step 7 — Write the two implementations

The catalog holds the schemas and no bodies. This cell is the bodies.

**Your app.** `finance_research` is two HTTP calls, in this order and not the other:

1. A `GET` to `fc.yahoo.com`, whose only job is to leave a session cookie on the session.
2. A `POST` to Yahoo's own news-stream API, which needs that cookie.

Both need a browser-shaped `User-Agent`. This is exactly what
`langchain_community`'s `YahooFinanceNewsTool` does internally; here it is in eight lines you can
read.

It is also the reason this tool has to be a `client` executor rather than an HTTP one, from
Step 3: a request the gateway can describe cannot carry a cookie it picked up from a different
request first.

The headlines change every day, so the output saved in this notebook is not what you will get.

In [10]:
from datetime import datetime, timezone

YAHOO_NEWS = "https://finance.yahoo.com/xhr/ncp?queryRef=latestNews&serviceKey=ncp_fin"


def finance_research_impl(ticker_symbol: str, limit: int = 3) -> str:
    """Recent Yahoo Finance headlines for one ticker, as plain text for the model."""
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    session.get("https://fc.yahoo.com", timeout=15)          # 1. pick up the session cookie
    res = session.post(                                       # 2. the real news call
        YAHOO_NEWS,
        json={"serviceConfig": {"snippetCount": limit, "s": [ticker_symbol]}},
        timeout=20,
    )
    res.raise_for_status()
    stream = res.json()["data"]["tickerStream"]["stream"]
    items = [
        f"{item['content']['title']}\n{item['content'].get('summary') or ''}".strip()
        for item in stream[:limit]
    ]
    return "\n\n".join(items) or f"No recent news found for {ticker_symbol}."


def todays_date_impl() -> str:
    """Today's date. A model cannot know this - it was trained in the past."""
    return datetime.now().strftime("%Y-%m-%d")


#: Keyed by catalog tool name. The catalog holds the schemas and deliberately no bodies,
#: so this map is the whole of what your app contributes.
IMPLEMENTATIONS = {NEWS_TOOL: finance_research_impl, DATE_TOOL: todays_date_impl}


def run_tool(name: str, args: dict) -> str:
    """Route one tool call from the model to its local implementation.

    `args` is often {} - get_todays_date takes nothing - so this unpacks rather than
    indexing. Indexing an expected key is how a zero-argument tool breaks a loop.
    """
    if name not in IMPLEMENTATIONS:
        raise ValueError(f"the model asked for an unknown tool: {name}")
    return IMPLEMENTATIONS[name](**args)


print("today:", run_tool(DATE_TOOL, {}))
print("\nAAPL news, first item only:")
print(run_tool(NEWS_TOOL, {"ticker_symbol": "AAPL"}).split("\n\n")[0])

today: 2026-08-22

AAPL news, first item only:
Anthropic's IPO could come sooner than you think — likely beating OpenAI to the punch
Anthropic's (ANTH.PVT) IPO paperwork could be filed as soon as this month, potentially beating OpenAI (OPAI.PVT) to the punch.  Yahoo Finance Technology Editor Dan Howley outlines what we know so far.


---

## Step 8 — Report the spans yourself

**Your app.** This is the part that only exists on the BYO path. Two functions, one per kind of
span, both posting to the same `POST /traces` endpoint and both carrying the same `traceId`.

`report_llm_span` is the one the gateway would have written for you. Note where each value comes
from: `model` from OpenAI's *response* rather than your constant, because a provider can answer
with a dated build of the model you asked for; `usage` from OpenAI's own `usage` object; and
`promptVersionId` from the render, which is what makes the run findable from the prompt version
later.

`capturePayloads: True` is what stores the `input` and `output`. Without it the spans still
appear, with nothing in them.

In [11]:
import uuid


def now() -> str:
    """Current time as an ISO-8601 string with a timezone offset, which is what /traces wants."""
    return datetime.now(timezone.utc).isoformat()


def report_llm_span(trace_id, *, model, started, ended, usage, prompt_version_id,
                    messages, output_message):
    """Report one `llm` span. On the BYO path there is no gateway to do it for us."""
    api("POST", "/traces", {"traces": [{
        "traceId": trace_id,
        "name": PROMPT,
        "capturePayloads": True,
        "spans": [{
            "spanId": f"llm-{uuid.uuid4()}",
            "name": model,
            "kind": "llm",
            "status": "ok",
            "startTime": started,
            "endTime": ended,
            "model": model,
            "provider": PROVIDER_HOST,           # the host really called, not the gateway
            "usage": {
                "promptTokens": usage.get("prompt_tokens"),
                "completionTokens": usage.get("completion_tokens"),
                "totalTokens": usage.get("total_tokens"),
            },
            "promptVersionId": prompt_version_id,   # links the run back to the version
            "input": {"messages": messages},
            "output": output_message,
        }],
    }]})


def report_tool_span(trace_id, *, name, args, result, started, ended):
    """Report one `tool` span. Yours on every path, because your process ran the tool."""
    api("POST", "/traces", {"traces": [{
        "traceId": trace_id,
        "capturePayloads": True,
        "spans": [{
            "spanId": f"{name}-{started}",
            "name": name,
            "kind": "tool",
            "status": "ok",
            "startTime": started,
            "endTime": ended,
            "input": args,
            "output": {"text": result},
        }],
    }]})


print("span reporters ready")

span reporters ready


---

## Step 9 — Run the agent

**Your app.** The loop, and the thing the whole notebook builds up to.

One detail is different from every gateway-based tutorial: the trace id. On the gateway path the
gateway hands you one back in a header. Here nobody does, so **you mint it yourself** with
`uuid4()` before the first call and use it on every span.

Then it is five moves, repeating:

1. Call OpenAI directly with the rendered messages and tools.
2. Report the `llm` span for that call.
3. If the model asked for no tools, that is the answer — stop.
4. Otherwise run each tool, report its `tool` span, and append the result as a `tool` message with
   the `tool_call_id` that pairs it to the request.
5. Go back to 1.

The turn cap stops a misbehaving model looping forever. Keep one in your own code.

In [12]:
def complete(model: str, messages: list, tools: list) -> dict:
    """One completion sent straight to the provider - never through AcruxCore."""
    res = requests.post(
        f"{PROVIDER_URL}/chat/completions",
        headers={"Authorization": f"Bearer {PROVIDER_KEY}", "Content-Type": "application/json"},
        json={"model": model, "messages": messages, "tools": tools},
        timeout=120,
    )
    res.raise_for_status()
    return res.json()


def ask(question: str, max_turns: int = 5) -> tuple[str, str]:
    """Answer one question with the stored prompt, both tools, and the provider direct."""
    rendered = api("POST", f"/prompts/{PROMPT}/production/render",
                   {"variables": {"question": question}})
    messages, tools = rendered["messages"], rendered["tools"]
    version_id = rendered["versionId"]

    trace_id = str(uuid.uuid4())      # BYO: no gateway trace to adopt, so mint our own
    for turn in range(1, max_turns + 1):
        started = now()
        data = complete(MODEL, messages, tools)
        ended = now()

        message = data["choices"][0]["message"]
        report_llm_span(
            trace_id,
            model=data["model"],          # what actually answered, e.g. a dated build
            started=started, ended=ended,
            usage=data.get("usage") or {},
            prompt_version_id=version_id,
            messages=messages, output_message=message,
        )
        messages.append(message)

        tool_calls = message.get("tool_calls")
        if not tool_calls:
            print(f"\n({turn} model turn(s), trace {trace_id})")
            return message["content"], trace_id

        for call in tool_calls:
            name = call["function"]["name"]
            args = json.loads(call["function"]["arguments"])
            t_started = now()
            result = run_tool(name, args)
            t_ended = now()
            print(f"  -> {name}({args})")
            print(f"     {result.splitlines()[0][:110]}")
            report_tool_span(trace_id, name=name, args=args, result=result,
                             started=t_started, ended=t_ended)
            messages.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "content": json.dumps(result),
            })

    raise RuntimeError("hit the turn limit without a final answer")


print("Q:", QUESTION)
answer, TRACE_ID = ask(QUESTION)
print("\nAssistant:", answer)

Q: Is there any recent news on AAPL, and is today a weekday?
  -> finance_research({'ticker_symbol': 'AAPL'})
     Anthropic's IPO could come sooner than you think — likely beating OpenAI to the punch
  -> get_todays_date({})
     2026-08-22

(2 model turn(s), trace 8495d81b-4bda-4ba9-a45d-b4603d2ad784)

Assistant: Here's the recent news regarding Apple Inc. (AAPL):

1. **Job Cuts for AI Focus**: Apple has reportedly cut more than 200 jobs across teams associated with Siri, software engineering, and Vision Pro. This shift aims to concentrate on artificial intelligence developments and smart glasses technology.

As for today's date, it is August 22, 2026, which falls on a weekday (Tuesday).

In summary, there is recent news about Apple focusing more on AI and smart glasses by cutting jobs, and today is indeed a weekday.


Read the tool lines. The model was asked one question and decided on its own that it needed
**two** different lookups, then combined both into one answer. Nothing in the code chose the
tools, the order, or the ticker.

That is what the ReAct instructions in the system message bought: not new capability, but a model
that stops to work out what the question needs.

But look at the answer again before you trust it.

**Check.** The question had two halves, and the tools only answered one of them each. The date
tool returned a date. Turning that date into a weekday was arithmetic the **model** did, with no
tool involved — and arithmetic is the thing models are worst at.

This cell checks that step against Python. Whether it agrees on your run or not, the lesson is
the same: a value that came from a tool is grounded, and a value the model derived from it is not.

In [13]:
true_date = datetime.now()
true_weekday = true_date.strftime("%A")
is_weekday = true_date.weekday() < 5

named = [day for day in
         ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
         if day.lower() in answer.lower()]

print(f"the tool returned:        {true_date.strftime('%Y-%m-%d')}")
print(f"that date really is a:    {true_weekday}  (a weekday: {is_weekday})")
print(f"weekday named in the answer: {named or 'none'}")
print(f"the model got it right:   {named == [true_weekday]}")
print("\nIf that last line is False, the model read the tool correctly and then did"
      " the arithmetic wrong. If the weekday matters, return it from the tool\n"
      "instead of letting the model derive it.")

the tool returned:        2026-08-22
that date really is a:    Saturday  (a weekday: False)
weekday named in the answer: ['Tuesday']
the model got it right:   False

If that last line is False, the model read the tool correctly and then did the arithmetic wrong. If the weekday matters, return it from the tool
instead of letting the model derive it.


That check passes on some runs and fails on others, with the same prompt and the same tool
output. The date is always right, because a tool produced it. The weekday is a coin toss, because
the model derived it.

That is the honest limit of a ReAct agent. The tools removed the unknown — *what is today* — and
the model still had to reason from it. Every derived value is a place to check, and the cheapest
fix is almost always to move the derivation into the tool: have `get_todays_date` return
`{"date": "2026-08-21", "weekday": "Friday"}` and there is no arithmetic left to get wrong.

---

## Step 10 — Read the trace back

**Check.** `GET /traces/<id>` returns the trace header plus every span. Two things to look for,
and only one of them is the span list.

The **shape** is the obvious one: model turns and tool calls, alternating, all under one trace id
you minted yourself.

The other is `costUsd`. It is empty on every span, and that is correct rather than broken. Cost is
computed from the prices attached to a gateway model, and this call never went near the gateway.
If you need cost on the BYO path, you compute it from `usage` yourself.

Token counts, timings and today's headlines all move between runs. Only the shape is stable.

In [14]:
detail = api("GET", f"/traces/{TRACE_ID}")


def walk(spans, depth=0):
    for span in spans:
        cost = "-" if span.get("costUsd") in (None, 0) else span["costUsd"]
        print(f"{'  ' * depth}- [{span['kind']}] {span['name'][:34]:<34} "
              f"{span['status']}  tokens={span.get('totalTokens') or 0:<5} cost={cost}")
        walk(span.get("children") or [], depth + 1)


print(f"trace:  {detail['trace']['name']}")
print(f"spans:  {detail['trace']['spanCount']}   "
      f"tokens: {detail['trace']['totalTokens']}   "
      f"cost: {detail['trace']['totalCostUsd']}")
walk(detail["spans"])

llm_spans = [s for s in detail["spans"] if s["kind"] == "llm"]
print(f"\nprovider on the llm spans: {sorted({s['provider'] for s in llm_spans})}")
print(f"prompt version stamped on them: "
      f"{sorted({s['promptVersionId'] for s in llm_spans})}")

trace:  react-agent-finance
spans:  4   tokens: 805   cost: None
- [llm] gpt-4o-mini-2024-07-18             ok  tokens=227   cost=-
- [tool] finance_research                   ok  tokens=0     cost=-
- [tool] get_todays_date                    ok  tokens=0     cost=-
- [llm] gpt-4o-mini-2024-07-18             ok  tokens=578   cost=-

provider on the llm spans: ['api.openai.com']
prompt version stamped on them: ['2807da05-b3bb-4032-96ad-6ebdc2c7be77']


That last line is what `promptVersionId` bought. Because every `llm` span carries it, the trace is
reachable from the prompt version as well as from the trace list — `GET
/prompts/<id>/versions/<n>/traces` answers "which runs used this exact version?".

The dashboard shows the same thing under **Observability → Traces**:

![The traces list showing react-agent-finance runs](https://docs.acruxcore.com/img/tutorials/build-a-react-agent/02-traces-list.png)

![A react-agent-finance trace expanded, showing the alternating model turns and tool calls](https://docs.acruxcore.com/img/tutorials/build-a-react-agent/03-trace-detail.png)

**Check.** Ask that question directly. The lineage route is all-time and applies no date window,
so it answers for the whole history of the version rather than the last thirty days.

In [15]:
lineage = api("GET", f"/prompts/{prompt['id']}/versions/1/traces?limit=5")
print(f"traces that used v1 of this prompt: {lineage['total']}")
for row in lineage["data"][:5]:
    print(f"  {row['id']}  spans={row['spanCount']}")

traces that used v1 of this prompt: 1
  8495d81b-4bda-4ba9-a45d-b4603d2ad784  spans=4


---

## Step 11 — Four ways to get this wrong

Every cell in this step is **broken on purpose**. None of it is app code.

### Mistake 1 — you never report the `llm` span

**Broken on purpose.** The quiet one, and the reason Step 1 exists. Run the loop, skip the span,
and everything still works from the outside. The trace even exists, because the tool span created
it. It just has no model turns in it.

In [16]:
rendered = api("POST", f"/prompts/{PROMPT}/production/render",
               {"variables": {"question": "Any news on MSFT?"}})
messages, tools = rendered["messages"], rendered["tools"]

silent_trace = str(uuid.uuid4())
data = complete(MODEL, messages, tools)
message = data["choices"][0]["message"]
# Broken on purpose: no report_llm_span call here at all.

for call in message.get("tool_calls") or []:
    args = json.loads(call["function"]["arguments"])
    started = now()
    report_tool_span(silent_trace, name=call["function"]["name"], args=args,
                     result=run_tool(call["function"]["name"], args),
                     started=started, ended=now())

detail = api("GET", f"/traces/{silent_trace}")
kinds = [s["kind"] for s in detail["spans"]]
print(f"spans in the trace: {detail['trace']['spanCount']}  kinds={kinds}")
print(f"tokens recorded:    {detail['trace']['totalTokens']}")
print("\nA tool ran, and nothing says a model asked for it.")

spans in the trace: 1  kinds=['tool']
tokens recorded:    0

A tool ran, and nothing says a model asked for it.


No error, no warning, and the OpenAI call still cost you money. From the trace alone you cannot
tell which model ran, how many tokens it used, or which prompt version produced it.

### Mistake 2 — a key sent to the wrong host

**Broken on purpose.** Two keys going to two different places is the shape of the BYO path, and
swapping them is the most common mistake on it. Both directions fail, and the body tells you which
way round you got it.

One thing the cell deliberately does *not* print: OpenAI's error **message**. It echoes the key
you sent with the middle masked, which still leaves the prefix and the last characters in your
logs. Print the `type` and `code` instead. That habit is worth keeping for every provider error
you log.

In [17]:
# AcruxCore key sent to the provider.
res = requests.post(
    f"{PROVIDER_URL}/chat/completions",
    headers={"Authorization": f"Bearer {ACRUX_KEY}", "Content-Type": "application/json"},
    json={"model": MODEL, "messages": [{"role": "user", "content": "hi"}]},
    timeout=30,
)
error = res.json().get("error") or {}
# Print the type and code, never the message: the provider echoes a partly-masked copy of the
# key you sent, and this notebook's output is committed to a repository.
print(f"acruxcore key -> {PROVIDER_HOST}:  HTTP {res.status_code}  "
      f"{error.get('type')} / {error.get('code')}")

# Provider key sent to AcruxCore.
res = requests.get(f"{ACRUX_URL}/prompts?limit=1",
                   headers={"Authorization": f"Bearer {PROVIDER_KEY}"}, timeout=30)
print(f"\nprovider key -> acruxcore: HTTP {res.status_code}")
print("  ", json.dumps(res.json())[:160])

acruxcore key -> api.openai.com:  HTTP 401  invalid_request_error / invalid_api_key

provider key -> acruxcore: HTTP 401
   {"error": {"code": "UNAUTHORIZED", "message": "Invalid or revoked API key."}}


### Mistake 3 — assuming every tool call has arguments

**Broken on purpose.** `get_todays_date` takes nothing, so the model sends `arguments` as the
string `"{}"`, which parses to an empty dict. Code that reaches into that dict for a key raises,
and it raises inside your loop rather than at the API boundary — which is a harder place to see
it.

In [18]:
call_with_no_args = {"function": {"name": DATE_TOOL, "arguments": "{}"}}
args = json.loads(call_with_no_args["function"]["arguments"])
print("the model sent:", repr(args))

try:
    # Broken on purpose: indexing a key an argument-free tool will never send.
    print(IMPLEMENTATIONS[DATE_TOOL](args["ticker_symbol"]))
except KeyError as err:
    print(f"KeyError: {err}")
    print("fix: unpack with **args, as run_tool does, so a zero-argument tool just works")

print("\nunpacked instead:", run_tool(DATE_TOOL, args))

the model sent: {}
KeyError: 'ticker_symbol'
fix: unpack with **args, as run_tool does, so a zero-argument tool just works

unpacked instead: 2026-08-22


### Mistake 4 — the span carries no `promptVersionId`

**Broken on purpose.** This one costs you nothing today and everything later. The span is stored,
the trace reads fine, and the run simply never appears when you ask which traces used a given
prompt version. That question is how you tell whether a prompt change made things better or
worse.

In [19]:
unstamped = str(uuid.uuid4())
started = now()
api("POST", "/traces", {"traces": [{
    "traceId": unstamped,
    "name": PROMPT,
    "capturePayloads": True,
    "spans": [{
        "spanId": f"llm-{uuid.uuid4()}",
        "name": MODEL, "kind": "llm", "status": "ok",
        "startTime": started, "endTime": now(),
        "model": MODEL, "provider": PROVIDER_HOST,
        "usage": {"promptTokens": 10, "completionTokens": 5, "totalTokens": 15},
        # Broken on purpose: no promptVersionId key.
    }],
}]})

print("the span stored fine:",
      api("GET", f"/traces/{unstamped}")["trace"]["spanCount"], "span")

lineage = api("GET", f"/prompts/{prompt['id']}/versions/1/traces?limit=100")
ids = {row["id"] for row in lineage["data"]}
print(f"\ntraces the lineage endpoint knows about: {lineage['total']}")
print(f"  the properly stamped run is there: {TRACE_ID in ids}")
print(f"  this unstamped one is there:       {unstamped in ids}")

the span stored fine: 1 span

traces the lineage endpoint knows about: 1
  the properly stamped run is there: True
  this unstamped one is there:       False


---

## Step 12 — Close the session

**Your app.** Worth knowing what this does and does not do.

With the SDK there is a background queue: spans are reported off the critical path so they never
slow your request down, and `await hub.gateway.aclose()` flushes whatever is still waiting. There
is no queue here. Every `POST /traces` above was synchronous and already finished, so there is
nothing to flush.

What is left is the TCP connections the two sessions are holding. Close them.

In [20]:
acrux.close()
yahoo.close()
print("sessions closed")

sessions closed


---

## What you built

A finance agent that reasons about what a question needs, calls two real tools to find out, and
answers from what they returned — with the model call going straight to your provider, and the trace
still complete because you wrote the model turns down yourself.

### What of this actually ships

The loop, the two implementations, and the two span reporters:

```python
import json, os, requests, uuid
from datetime import datetime, timezone
from urllib.parse import urlsplit

ACRUX_URL = os.environ["ACRUXCORE_BASE_URL"].rstrip("/")
acrux = requests.Session()
acrux.headers.update({"Authorization": f"Bearer {os.environ['ACRUXCORE_API_KEY']}",
                      "Content-Type": "application/json"})
PROVIDER_URL = os.environ["PROVIDER_BASE_URL"].rstrip("/")   # any OpenAI-compatible API
PROVIDER_KEY, MODEL = os.environ["PROVIDER_API_KEY"], "gpt-4o-mini"
PROVIDER_HOST = urlsplit(PROVIDER_URL).netloc


def ask(question, max_turns=5):
    rendered = acrux.post(f"{ACRUX_URL}/prompts/react-agent-finance/production/render",
                          json={"variables": {"question": question}}).json()
    messages, tools = rendered["messages"], rendered["tools"]
    trace_id = str(uuid.uuid4())            # BYO: nobody hands you one

    for _ in range(max_turns):
        started = datetime.now(timezone.utc).isoformat()
        data = requests.post(f"{PROVIDER_URL}/chat/completions",
                             headers={"Authorization": f"Bearer {PROVIDER_KEY}"},
                             json={"model": MODEL, "messages": messages,
                                   "tools": tools}).json()
        message, usage = data["choices"][0]["message"], data.get("usage") or {}

        acrux.post(f"{ACRUX_URL}/traces", json={"traces": [{
            "traceId": trace_id, "name": "react-agent-finance", "capturePayloads": True,
            "spans": [{"spanId": f"llm-{uuid.uuid4()}", "name": data["model"], "kind": "llm",
                       "status": "ok", "startTime": started,
                       "endTime": datetime.now(timezone.utc).isoformat(),
                       "model": data["model"], "provider": PROVIDER_HOST,
                       "usage": {"promptTokens": usage.get("prompt_tokens"),
                                 "completionTokens": usage.get("completion_tokens"),
                                 "totalTokens": usage.get("total_tokens")},
                       "promptVersionId": rendered["versionId"],
                       "input": {"messages": messages}, "output": message}]}]})
        messages.append(message)

        if not message.get("tool_calls"):
            return message["content"]

        for call in message["tool_calls"]:
            args = json.loads(call["function"]["arguments"])
            result = IMPLEMENTATIONS[call["function"]["name"]](**args)   # from Step 7
            # ... report the tool span the same way, then:
            messages.append({"role": "tool", "tool_call_id": call["id"],
                             "content": json.dumps(result)})

    raise RuntimeError("hit the turn limit")
```

Everything else was scaffolding:

- `api`, `find_by_name` and `create_tool_if_missing` exist so this notebook can be re-run. They
  are notebook helpers, not an SDK.
- the create-and-commit cells are the dashboard's job, done once.
- every **Check** cell — the preflight, `resolve`, `render`, the trace walk, the lineage call —
  proves a step worked. None of it belongs in a request path.
- Step 11 is all deliberately broken.

### What this notebook left in your team

- two tools at v1, `finance_research` and `get_todays_date`, both `client` executors
- a prompt `react-agent-finance` at v1, with a `question` variable and **no** bound model
- two bindings, inherited by every prompt alias
- several traces: the good run, one with no model turns from Mistake 1, and one unstamped span

### Where to go next

- [Build a RAG agent without the gateway](https://docs.acruxcore.com/docs/tutorials/build-a-rag-agent-without-the-gateway)
  — the same BYO idea, with the SDK's `provider=` option doing the span reporting for you.
- [Build a tool-calling agent in Python (SDK)](https://docs.acruxcore.com/docs/tutorials/build-a-tool-calling-agent-in-python-sdk)
  — the gateway path, where the `llm` spans are written for you.
- [Using sessions and traces](https://docs.acruxcore.com/docs/guides/using-sessions-and-traces)
  — group related runs and dig into what happened.